In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PATH = PROJECT_ROOT / "data" / "bitext_customer_support_raw.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [3]:
# Missing values
print("Missing values per column:")
print(df.isnull().sum())

# Category distribution
print("\nCategory counts:")
print(df["category"].value_counts())

# Intent distribution (just how many unique intents, and min/max count per intent)
print("\nNumber of unique intents:", df["intent"].nunique())
print(df["intent"].value_counts().describe())

Missing values per column:
flags          0
instruction    0
category       0
intent         0
response       0
dtype: int64

Category counts:
category
ACCOUNT         5986
ORDER           3988
REFUND          2992
INVOICE         1999
CONTACT         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64

Number of unique intents: 27
count      27.000000
mean      995.259259
std        10.368564
min       950.000000
25%       997.000000
50%       998.000000
75%       999.000000
max      1000.000000
Name: count, dtype: float64


In [4]:
# Duplicate rows
print("Duplicate instructions:", df["instruction"].duplicated().sum())

# Text length (in words) for instruction and response
df["instruction_len"] = df["instruction"].str.split().str.len()
df["response_len"] = df["response"].str.split().str.len()

print("\nInstruction length stats:")
print(df["instruction_len"].describe())

print("\nResponse length stats:")
print(df["response_len"].describe())

Duplicate instructions: 2237

Instruction length stats:
count    26872.000000
mean         8.690979
std          2.605004
min          1.000000
25%          7.000000
50%          9.000000
75%         11.000000
max         16.000000
Name: instruction_len, dtype: float64

Response length stats:
count    26872.000000
mean       104.789037
std         52.966204
min          9.000000
25%         72.000000
50%         90.000000
75%        124.000000
max        402.000000
Name: response_len, dtype: float64


In [5]:
# Check whether duplicated instructions have consistent labels or conflicting ones
dup_instructions = df[df["instruction"].duplicated(keep=False)]
conflict_check = dup_instructions.groupby("instruction")["intent"].nunique()
print("Duplicated instructions with more than one distinct intent:", (conflict_check > 1).sum())
print("Duplicated instructions with a single consistent intent:", (conflict_check == 1).sum())

Duplicated instructions with more than one distinct intent: 0
Duplicated instructions with a single consistent intent: 989


In [6]:
df_clean = df.drop_duplicates(subset=["instruction"], keep="first").reset_index(drop=True)
print("Rows before:", len(df))
print("Rows after removing duplicates:", len(df_clean))

Rows before: 26872
Rows after removing duplicates: 24635


In [7]:
CLEAN_FILE = PROJECT_ROOT / "data" / "bitext_customer_support_clean.csv"
df_clean.to_csv(CLEAN_FILE, index=False)
print(f"Saved clean dataset to {CLEAN_FILE}")

Saved clean dataset to C:\Users\RayanSystem\Dropbox\Pers\Python\Python405\Projects\GitHub Repositories\Customer Support Ticket Intelligence\data\bitext_customer_support_clean.csv
